## Prepare shortage panel

### below code computes prepares demand and computes shortage

In [3]:
"""
Combine NITI Aayog ICED state-level electricity Excel files into a single
monthly panel: state | state_name | year | month (1-12) | state_month |
state_year | peak_demand | peak_demand_met | electricity_requirement |
electricity_supplied | peak_shortage | electricity_shortage |
relative_electricity_shortage (%) | detrended_electricity_requirement

Folder structure expected (relative to DATA_DIR):
    peak_demand/PD_XX.xlsx
    peak_demand_met/PDM_XX.xlsx
    electricity_requirement/ER_XX.xlsx
    electricity_supplied/ES_XX.xlsx
    list_of_states.xlsx

Each file is wide format: a "Parameter" header row with YYYY-MM month
columns, one value row, and a copyright footer.

"""

import calendar
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('data/prep_panel')
OUTPUT_CSV = DATA_DIR / "electricity_panel.csv"
OUTPUT_PLOT = DATA_DIR / "detrended_requirement_by_month.png"

# folder name -> (file prefix, output column name)
SOURCES = {
    "peak_demand": ("PD", "peak_demand"),
    "peak_demand_met": ("PDM", "peak_demand_met"),
    "electricity_requirement": ("ER", "electricity_requirement"),
    "electricity_supplied": ("ES", "electricity_supplied"),
}

# derived column -> (minuend, subtrahend)
DERIVED = {
    "peak_shortage": ("peak_demand", "peak_demand_met"),
    "electricity_shortage": ("electricity_requirement", "electricity_supplied"),
}

DETRENDED_COL = "detrended_electricity_requirement"


def read_wide_file(path: Path) -> pd.Series:
    """Read one ICED wide-format file; return Series indexed by month Period."""
    raw = pd.read_excel(path, header=None, dtype=object)

    # Locate the header row containing "Parameter"
    header_idx = None
    for i, row in raw.iterrows():
        if row.astype(str).str.strip().str.lower().eq("parameter").any():
            header_idx = i
            break
    if header_idx is None:
        raise ValueError(f"No 'Parameter' header row found in {path.name}")

    header = raw.iloc[header_idx].astype(str).str.strip()
    values = raw.iloc[header_idx + 1]

    # Keep only columns whose header parses as YYYY-MM
    months, vals = [], []
    for col, name in header.items():
        try:
            period = pd.Period(name, freq="M")
        except Exception:
            continue
        months.append(period)
        vals.append(pd.to_numeric(values[col], errors="coerce"))

    if not months:
        raise ValueError(f"No month columns found in {path.name}")

    s = pd.Series(vals, index=pd.PeriodIndex(months, name="month"))
    # Guard against duplicated month columns
    s = s[~s.index.duplicated(keep="first")]
    return s.sort_index()


def load_variable(folder: str, prefix: str, colname: str) -> pd.DataFrame:
    """Read every state file for one variable; return long df: state, month, value."""
    files = sorted((DATA_DIR / folder).glob(f"{prefix}_*.xlsx"))
    if not files:
        print(f"WARNING: no {prefix}_*.xlsx files found in {DATA_DIR / folder}")
        return pd.DataFrame(columns=["state", "month", colname])

    frames = []
    for f in files:
        state_code = f.stem.split("_", 1)[1].upper()
        try:
            s = read_wide_file(f)
        except Exception as e:
            print(f"WARNING: skipping {f.name}: {e}")
            continue
        frames.append(
            pd.DataFrame({"state": state_code, "month": s.index, colname: s.values})
        )
    print(f"{colname}: read {len(frames)} state files from {folder}/")
    return pd.concat(frames, ignore_index=True)


def load_state_names() -> pd.DataFrame | None:
    """Map state codes to full names using list_of_states.xlsx (no header; name, code)."""
    path = DATA_DIR / "list_of_states.xlsx"
    if not path.exists():
        return None
    df = pd.read_excel(path, header=None, names=["state_name", "state"], dtype=str)
    df["state"] = df["state"].str.strip().str.upper()
    # Underscores instead of spaces so names are safe as IDs/factor levels
    df["state_name"] = df["state_name"].str.strip().str.replace(" ", "_")
    return df[["state", "state_name"]].dropna()


def add_detrended_requirement(panel: pd.DataFrame) -> None:
    """Add `detrended_electricity_requirement` in place.

    Within each (state, calendar-month) cell, electricity_requirement is
    regressed on `year` with an ordinary least-squares linear fit, and the
    detrended value is the residual (actual minus fitted). This removes each
    calendar month's own year-on-year growth trend for that state, leaving the
    interannual deviations around that trend (in the same units as the raw
    requirement). Cells with fewer than two valid observations are left NaN.

    Detrending is done per state as well as per month so that each state's
    trend is estimated on its own magnitude; to instead detrend a single
    pooled month-of-year trend across all states, group by "month" only.
    """
    detrended = pd.Series(np.nan, index=panel.index, dtype="float64")

    for _, idx in panel.groupby(["state", "month"]).groups.items():
        sub = panel.loc[idx]
        x = sub["year"].to_numpy(dtype="float64")
        y = sub["electricity_requirement"].to_numpy(dtype="float64")
        ok = np.isfinite(x) & np.isfinite(y)
        # need at least two points AND some variation in year to fit a trend
        if ok.sum() >= 2 and np.ptp(x[ok]) > 0:
            slope, intercept = np.polyfit(x[ok], y[ok], 1)
            resid = y - (slope * x + intercept)
            detrended.loc[idx] = np.where(ok, resid, np.nan)

    panel[DETRENDED_COL] = detrended


def plot_detrended_by_month(panel: pd.DataFrame, path: Path) -> None:
    """Grid of 12 line charts (one per calendar month) of detrended requirement.

    Within each subplot, one line per state shows the detrended requirement
    across years (deviations around each state's month-specific trend).
    """
    fig, axes = plt.subplots(4, 3, figsize=(16, 14), sharex=True)
    axes = axes.ravel()

    for m in range(1, 13):
        ax = axes[m - 1]
        sub = panel[panel["month"] == m]
        for _, g in sub.groupby("state"):
            g = g.sort_values("year")
            ax.plot(g["year"], g[DETRENDED_COL], linewidth=0.8, alpha=0.6)
        ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.7)
        ax.set_title(calendar.month_name[m], fontsize=11)
        ax.grid(True, alpha=0.3)

    for i, ax in enumerate(axes):
        if i % 3 == 0:
            ax.set_ylabel("Detrended requirement")
        if i >= 9:
            ax.set_xlabel("Year")

    fig.suptitle(
        "Month-of-year detrended electricity requirement by state",
        fontsize=14, y=0.995,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Plot written to {path}")
    plt.show()   # render inline in a notebook (harmless in a plain script)
    return fig


def main() -> None:
    panel = None
    for folder, (prefix, colname) in SOURCES.items():
        df = load_variable(folder, prefix, colname)
        panel = df if panel is None else panel.merge(df, on=["state", "month"], how="outer")

    if panel is None or panel.empty:
        sys.exit("No data read - check DATA_DIR and folder names.")

    # Derived shortage columns (NaN if either input is missing)
    for colname, (minuend, subtrahend) in DERIVED.items():
        panel[colname] = panel[minuend] - panel[subtrahend]
        n_negative = (panel[colname] < 0).sum()
        if n_negative:
            print(f"NOTE: {colname} has {n_negative} negative values "
                  f"(supply/met exceeds demand/requirement in source data)")

    # Relative shortage: % of requirement unmet (NaN if requirement missing or zero)
    requirement = panel["electricity_requirement"].where(panel["electricity_requirement"] != 0)
    panel["relative_electricity_shortage"] = (
        100 * panel["electricity_shortage"] / requirement
    )

    value_cols = ([col for _, col in SOURCES.values()]
                  + list(DERIVED)
                  + ["relative_electricity_shortage"])

    names = load_state_names()
    if names is not None:
        panel = panel.merge(names, on="state", how="left")
        unmatched = panel.loc[panel["state_name"].isna(), "state"].unique()
        if len(unmatched):
            print(f"WARNING: no state name match for codes: {sorted(unmatched)}")
        panel = panel[["state", "state_name", "month"] + value_cols]
    else:
        panel = panel[["state", "month"] + value_cols]

    panel = panel.sort_values(["state", "month"]).reset_index(drop=True)

    # Split Period month into numeric year and month (1-12)
    panel["year"] = panel["month"].dt.year
    panel["month"] = panel["month"].dt.month

    # Month-of-year detrended requirement (per state, per calendar month).
    # Done after year/month are numeric so the trend is fit on integer years.
    add_detrended_requirement(panel)
    value_cols = value_cols + [DETRENDED_COL]

    # Interaction IDs for fixed effects (use full state name if available)
    label = panel["state_name"] if "state_name" in panel.columns else panel["state"]
    panel["state_month"] = label + "_month" + panel["month"].astype(str)
    panel["state_year"] = label + "_" + panel["year"].astype(str)

    id_cols = [c for c in ("state", "state_name") if c in panel.columns]
    panel = panel[id_cols + ["year", "month", "state_month", "state_year"] + value_cols]
    panel.to_csv(OUTPUT_CSV, index=False)

    n_states = panel["state"].nunique()
    first = panel.loc[(panel["year"] * 100 + panel["month"]).idxmin()]
    last = panel.loc[(panel["year"] * 100 + panel["month"]).idxmax()]
    print(f"\nPanel written to {OUTPUT_CSV}")
    print(f"{len(panel):,} rows | {n_states} states | "
          f"{first['year']}-{first['month']:02d} to {last['year']}-{last['month']:02d}")
    print("\nMissing values per column:")
    print(panel.isna().sum().to_string())

    # Panel of per-month line graphs of the detrended requirement
    # plot_detrended_by_month(panel, OUTPUT_PLOT)


if __name__ == "__main__":
    main()

peak_demand: read 33 state files from peak_demand/
peak_demand_met: read 33 state files from peak_demand_met/
electricity_requirement: read 33 state files from electricity_requirement/
electricity_supplied: read 33 state files from electricity_supplied/
NOTE: peak_shortage has 66 negative values (supply/met exceeds demand/requirement in source data)
NOTE: electricity_shortage has 157 negative values (supply/met exceeds demand/requirement in source data)

Panel written to data/prep_panel/electricity_panel.csv
4,389 rows | 33 states | 2015-04 to 2026-04

Missing values per column:
state                                0
state_name                           0
year                                 0
month                                0
state_month                          0
state_year                           0
peak_demand                          0
peak_demand_met                      0
electricity_requirement              0
electricity_supplied                 0
peak_shortage           

## Below code prepares CDD data sources

In [4]:
"""
01_download_data.py
===================
Downloads everything needed for the CDD panel:

  1. ERA5-Land daily statistics from the CDS, one NetCDF file per
     within-year chunk, for an all-India bounding box:
       - daily-mean 2m temperature          (t2m)  -> dry-bulb CDD
       - daily-mean 2m dewpoint temperature (d2m)  -> RH and sWBGT CDD
  2. GADM v4.1 level-1 state boundaries for India.

GPWv4 population count must be downloaded MANUALLY (it sits behind a NASA
Earthdata login, so it can't be scripted without credentials):

    https://sedac.ciesin.columbia.edu/data/set/gpw-v4-population-count-rev11
    -> "Population Count, v4.11" -> GeoTIFF -> 2020 -> 30 arc-second
    -> file: gpw_v4_population_count_rev11_2020_30_sec.tif

Place that .tif in DATA_DIR/gpw/ (path configured below).

Requires: cdsapi, requests. Assumes ~/.cdsapirc is already configured.

Run this in one Jupyter cell; it is restart-safe (skips files that
already exist and are non-trivially sized). Existing t2m files keep the
same names as before, so nothing is re-downloaded; only the new d2m
chunks are fetched.
"""

import calendar
import zipfile
from pathlib import Path

import cdsapi
import requests

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DATA_DIR = Path("data/cdd_data")            # change to taste (relative to notebook)
ERA5_DIR = DATA_DIR / "era5_land_daily"
GADM_DIR = DATA_DIR / "gadm"
GPW_DIR = DATA_DIR / "gpw"             # put the GPW GeoTIFF here manually

START_YM = (2015, 4)                   # first panel month
END_YM = (2026, 4)                     # last panel month (inclusive)
CHUNK_MONTHS = 12                      # months per CDS request (within a year)

# short name used in filenames -> CDS variable name
# Requested as SEPARATE per-variable files: multi-variable requests to the
# derived daily-statistics datasets return a zip of per-variable NetCDFs,
# and separate files also keep the existing t2m cache valid.
VARIABLES = {
    "t2m": "2m_temperature",           # dry-bulb CDD
    "d2m": "2m_dewpoint_temperature",  # relative humidity + simplified WBGT
}

# All-India bounding box [N, W, S, E]. Covers every mainland state/UT in
# the panel (Andaman/Nicobar and Lakshadweep are not in the panel, so the
# box deliberately stops at 68E-98E).
AREA = [37.5, 67.5, 6.0, 98.0]

GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_IND_shp.zip"

for d in (ERA5_DIR, GADM_DIR, GPW_DIR):
    d.mkdir(parents=True, exist_ok=True)


def month_range(start_ym, end_ym):
    """Yield (year, month) tuples from start to end inclusive."""
    y, m = start_ym
    while (y, m) <= end_ym:
        yield y, m
        m += 1
        if m == 13:
            y, m = y + 1, 1


def month_chunks(start_ym, end_ym, size):
    """Yield lists of (year, month) tuples, chunked within calendar years.

    Chunks never straddle a year boundary, which keeps the CDS request
    simple (a single 'year' value) and the output filenames unambiguous.
    """
    chunk = []
    for y, m in month_range(start_ym, end_ym):
        if chunk and (chunk[0][0] != y or len(chunk) == size):
            yield chunk
            chunk = []
        chunk.append((y, m))
    if chunk:
        yield chunk


# ----------------------------------------------------------------------
# 1. ERA5-Land daily statistics: daily means, one variable per file
# ----------------------------------------------------------------------
def download_era5_land_daily(short_name, cds_variable):
    """Download daily-mean `cds_variable` in within-year chunks.

    Filenames follow the existing t2m pattern:
        {short_name}_daily_mean_{year}_{first:02d}-{last:02d}.nc
    so t2m files already on disk are recognised and skipped, and t2m/d2m
    chunk files pair up one-to-one by their year/month suffix.
    """
    client = cdsapi.Client()
    for chunk in month_chunks(START_YM, END_YM, CHUNK_MONTHS):
        year = chunk[0][0]
        months = [m for _, m in chunk]
        out = ERA5_DIR / (
            f"{short_name}_daily_mean_{year}_{months[0]:02d}-{months[-1]:02d}.nc"
        )
        if out.exists() and out.stat().st_size > 10_000:
            print(f"skip   {out.name} (already downloaded)")
            continue

        # Request days 01-31 for all months in the chunk; the CDS silently
        # skips day/month combinations that don't exist (e.g. 31 Feb).
        max_day = max(calendar.monthrange(year, m)[1] for m in months)
        request = {
            "variable": [cds_variable],
            "year": str(year),
            "month": [f"{m:02d}" for m in months],
            "day": [f"{d:02d}" for d in range(1, max_day + 1)],
            "daily_statistic": "daily_mean",
            # IST so that "a day" matches the local calendar day used by
            # the shortage reporting, rather than UTC days.
            "time_zone": "utc+05:30",
            "frequency": "1_hourly",
            "area": AREA,
        }
        print(f"fetch  {out.name} ...")
        client.retrieve("derived-era5-land-daily-statistics", request, str(out))
    print(f"ERA5-Land daily statistics ({short_name}): done.")


# ----------------------------------------------------------------------
# 2. GADM v4.1 level-1 boundaries for India
# ----------------------------------------------------------------------
def download_gadm():
    zip_path = GADM_DIR / "gadm41_IND_shp.zip"
    shp_path = GADM_DIR / "gadm41_IND_1.shp"
    if shp_path.exists():
        print("skip   GADM (already extracted)")
        return
    if not zip_path.exists():
        print("fetch  GADM v4.1 India shapefile ...")
        r = requests.get(GADM_URL, timeout=300)
        r.raise_for_status()
        zip_path.write_bytes(r.content)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(GADM_DIR)
    print("GADM: done.")


# ----------------------------------------------------------------------
if __name__ == "__main__" or True:   # "or True" so it runs directly in a cell
    for short_name, cds_variable in VARIABLES.items():
        download_era5_land_daily(short_name, cds_variable)
    download_gadm()

    gpw_file = GPW_DIR / "gpw-v4-population-count_2020.tif"
    if gpw_file.exists():
        print(f"GPW GeoTIFF found: {gpw_file.name}")
    else:
        print(
            "\nREMINDER: GPW population raster not found at "
            f"{gpw_file.resolve()}.\n"
            "Place gpw-v4-population-count_2020.tif there before running "
            "02_prepare_cdd_panel.py."
        )

skip   t2m_daily_mean_2015_04-12.nc (already downloaded)
skip   t2m_daily_mean_2016_01-12.nc (already downloaded)
skip   t2m_daily_mean_2017_01-12.nc (already downloaded)
skip   t2m_daily_mean_2018_01-12.nc (already downloaded)
skip   t2m_daily_mean_2019_01-12.nc (already downloaded)
skip   t2m_daily_mean_2020_01-12.nc (already downloaded)
skip   t2m_daily_mean_2021_01-12.nc (already downloaded)
skip   t2m_daily_mean_2022_01-12.nc (already downloaded)
skip   t2m_daily_mean_2023_01-12.nc (already downloaded)
skip   t2m_daily_mean_2024_01-12.nc (already downloaded)
skip   t2m_daily_mean_2025_01-12.nc (already downloaded)
skip   t2m_daily_mean_2026_01-04.nc (already downloaded)
ERA5-Land daily statistics (t2m): done.
skip   d2m_daily_mean_2015_04-12.nc (already downloaded)
skip   d2m_daily_mean_2016_01-12.nc (already downloaded)
skip   d2m_daily_mean_2017_01-12.nc (already downloaded)
skip   d2m_daily_mean_2018_01-12.nc (already downloaded)
skip   d2m_daily_mean_2019_01-12.nc (already dow

## Calculate CDDs, DP CDD, WBGT CDD, RH and complete panel

In [ ]:
"""
02_prepare_climate_panel.py  (redesigned cell 6)
================================================
Builds ALL population-weighted monthly climate exposures for the shortage
panel in one pass, on the GPWv4 / reproject-match architecture:

  cdd        : dry-bulb CDD,   base 25 degC   (sum of daily values)
  cdd_swbgt  : simplified-WBGT CDD, base 25   (sum;  Willett & Sherwood 2012)
  cdd_dewpt  : dew-point CDD,  base 21        (sum;  moisture-only exposure)
  rh         : relative humidity, %           (MEAN of daily values;
                                               Magnus/Bolton 1980)
  t_mean     : dry-bulb temperature, degC     (MEAN of daily-mean t2m;
                                               no base temperature)

Pipeline per variable (identical construction for every exposure, so the
regressors are directly comparable):
  1. Daily field at ERA5-Land grid from daily-mean t2m (and d2m where the
     variable needs moisture); degree days / RH computed at the grid-cell
     level BEFORE any aggregation.
  2. Aggregate days -> calendar month at the ERA5-Land grid
     (sum for degree days, mean for RH and t_mean; skipna=False so ocean
     stays NaN).
  3. Bilinear reproject onto the GPWv4 30 arc-second population grid.
  4. Population-weighted mean over dissolved panel-state geometries
     (Ladakh -> J&K; DNH + Daman & Diu merged).
  5. Merge onto the panel by (state_name, year, month).

Missing-file tolerance (d2m still downloading):
  * t2m chunks are the backbone. A month with t2m but no matching d2m file
    gets its dry-bulb CDD and t_mean now and NaN for the three humid
    variables.
  * Results are cached in CLIM_CACHE (state-month CSV). On re-run, a month
    is recomputed ONLY if it was never done, if its humid variables are
    still NaN and the d2m file has since appeared, or if t_mean is still
    NaN (e.g. the cache predates the t_mean column). So you can run this
    cell repeatedly while downloads trickle in; it fills the gaps and skips
    the rest. Delete CLIM_CACHE to force a full rebuild (e.g. after
    changing a base temperature). t_mean has no base temperature, so it
    never triggers cache invalidation on its own.

Panel format (from build_electricity_panel.py): state names use underscores
(e.g. Tamil_Nadu); time is integer `year` and `month` (1-12).

Run after cell 2 (needs OUTPUT_CSV in the kernel) and after 01_download has
at least started producing files.
"""

import re
from pathlib import Path
import json
import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  (registers the .rio accessor)
import xarray as xr
from rasterio.enums import Resampling
from rasterio.features import rasterize

# ----------------------------------------------------------------------
# Configuration -- keep consistent with 01_download_data.py
# ----------------------------------------------------------------------
DATA_DIR = Path("data/cdd_data")
ERA5_DIR = DATA_DIR / "era5_land_daily"
GADM_SHP = DATA_DIR / "gadm" / "gadm41_IND_1.shp"
GPW_TIF = DATA_DIR / "gpw" / "gpw-v4-population-count_2020.tif"

PANEL_CSV = OUTPUT_CSV     # from cell 2; adjust path if needed
OUT_CSV = Path("data/prep_panel/electricity_panel_with_cdd.csv")
CLIM_CACHE = DATA_DIR / "state_month_climate.csv"

DRYBULB_BASE_C = 26.0
SWBGT_BASE_C = 25.0       # NOT comparable to the dry-bulb base (humidity
                          # pushes sWBGT above T in the monsoon)
DEWPOINT_BASE_C = 21.0    # Td=25 would be ~all zeros; ~18-21 is the
                          # conventional oppressive-humidity threshold

CLIM_VARS = ["cdd", "t_mean", "cdd_swbgt", "cdd_dewpt", "rh"]
HUMID_VARS = ["cdd_swbgt", "cdd_dewpt", "rh"]      # need d2m
T2M_ONLY_VARS = ["cdd", "t_mean"]                  # computable from t2m alone

FNAME_RE = re.compile(r"t2m_daily_mean_(\d{4})_(\d{2})-(\d{2})\.nc")

CACHE_PARAMS = {"drybulb": DRYBULB_BASE_C,
                "swbgt": SWBGT_BASE_C,
                "dewpoint": DEWPOINT_BASE_C}
CACHE_META = CLIM_CACHE.with_suffix(".meta.json")

# GADM NAME_1 -> panel state_name (only names that differ; everything else
# is matched after normalisation that ignores spaces AND underscores).
GADM_TO_PANEL = {
    "NCT of Delhi": "Delhi",
    "Delhi": "Delhi",
    "Telangana": "Telengana",                 # panel spelling
    "Puducherry": "Puduchery",                # panel spelling
    "Ladakh": "Jammu_and_Kashmir",            # dissolve into J&K
    "Jammu and Kashmir": "Jammu_and_Kashmir",
    "Dadra and Nagar Haveli": "Dadra_and_Nagar_Haveli_and_Daman_and_Diu",
    "Daman and Diu": "Dadra_and_Nagar_Haveli_and_Daman_and_Diu",
    "Dadra and Nagar Haveli and Daman and Diu":
        "Dadra_and_Nagar_Haveli_and_Daman_and_Diu",
    "Orissa": "Odisha",
    "Uttaranchal": "Uttarakhand",
}


def _norm(s: str) -> str:
    return "".join(s.lower().replace("_", " ").split())


# ----------------------------------------------------------------------
# 1. Panel and state geometries
# ----------------------------------------------------------------------
panel = pd.read_csv(PANEL_CSV)
panel_states = sorted(panel["state_name"].unique())
panel_month_set = {(int(y), int(m))
                   for y, m in zip(panel["year"], panel["month"])}
print(f"Panel: {len(panel_states)} states, {len(panel_month_set)} months")

gadm = gpd.read_file(GADM_SHP)[["NAME_1", "geometry"]]
norm_panel = {_norm(s): s for s in panel_states}


def map_name(gadm_name: str):
    if gadm_name in GADM_TO_PANEL:
        mapped = GADM_TO_PANEL[gadm_name]
        return mapped if mapped in panel_states else None
    return norm_panel.get(_norm(gadm_name))


gadm["panel_state"] = gadm["NAME_1"].map(map_name)
unmatched_panel = sorted(set(panel_states) - set(gadm["panel_state"].dropna()))
if unmatched_panel:
    raise RuntimeError(f"Panel states with no GADM geometry: "
                       f"{unmatched_panel}. Add entries to GADM_TO_PANEL.")

states = (gadm.dropna(subset=["panel_state"])
              .dissolve(by="panel_state")
              .reset_index()[["panel_state", "geometry"]]
              .to_crs("EPSG:4326"))
state_names = list(states["panel_state"])
print(f"State geometries ready: {len(states)}")

# ----------------------------------------------------------------------
# 2. Population raster (GPWv4) and state label raster on its grid
# ----------------------------------------------------------------------
if not GPW_TIF.exists():
    raise FileNotFoundError(f"GPW GeoTIFF not found: {GPW_TIF.resolve()}")

minx, miny, maxx, maxy = states.total_bounds
pad = 0.5
pop = (rioxarray.open_rasterio(GPW_TIF, masked=True)
       .squeeze("band", drop=True)
       .rio.clip_box(minx - pad, miny - pad, maxx + pad, maxy + pad)
       .rio.write_crs("EPSG:4326"))
pop_np = pop.values.astype("float64")
pop_np[~np.isfinite(pop_np)] = 0.0
print(f"Population grid: {pop.shape}, total = {pop_np.sum() / 1e9:.3f} bn")

label = rasterize(
    [(geom, i + 1) for i, geom in enumerate(states.geometry)],
    out_shape=pop.shape, transform=pop.rio.transform(),
    fill=0, dtype="int32", all_touched=False,
)

# ----------------------------------------------------------------------
# 3. Helpers
# ----------------------------------------------------------------------
def open_var_degc(path: Path) -> xr.DataArray:
    """Open one single-variable ERA5-Land daily-statistics file -> degC,
    dims harmonised to (time, latitude, longitude)."""
    ds = xr.open_dataset(path)
    da = ds[next(iter(ds.data_vars))]
    rename = {}
    if "valid_time" in da.dims:
        rename["valid_time"] = "time"
    if "lat" in da.dims:
        rename["lat"] = "latitude"
    if "lon" in da.dims:
        rename["lon"] = "longitude"
    if rename:
        da = da.rename(rename)
    return da - 273.15


def sat_vp_hpa(t_degc):
    """Magnus saturation vapour pressure over water, hPa (Bolton 1980)."""
    return 6.112 * np.exp(17.67 * t_degc / (t_degc + 243.5))


def monthly_fields(path_t2m: Path, path_d2m: Path | None) -> dict:
    """All monthly exposure fields for one chunk file, at the ERA5-Land
    grid. Daily values are computed per grid cell FIRST, then aggregated
    to calendar months (skipna=False, so ocean NaNs propagate and are
    excluded from the weighted means downstream)."""
    t = open_var_degc(path_t2m)
    fields = {"cdd": (t - DRYBULB_BASE_C).clip(min=0.0)
                        .resample(time="MS").sum(skipna=False),
              "t_mean": t.resample(time="MS").mean(skipna=False)}
    if path_d2m is not None:
        td = open_var_degc(path_d2m)
        t_al, td_al = xr.align(t, td, join="inner")   # match timestamps
        e = sat_vp_hpa(td_al)                          # vapour pressure, hPa
        swbgt = 0.567 * t_al + 0.393 * e + 3.94        # simplified WBGT
        rh = (100.0 * e / sat_vp_hpa(t_al)).clip(0.0, 100.0)
        fields["cdd_swbgt"] = ((swbgt - SWBGT_BASE_C).clip(min=0.0)
                               .resample(time="MS").sum(skipna=False))
        fields["cdd_dewpt"] = ((td_al - DEWPOINT_BASE_C).clip(min=0.0)
                               .resample(time="MS").sum(skipna=False))
        fields["rh"] = rh.resample(time="MS").mean(skipna=False)
    return fields


def weighted_state_means(field: xr.DataArray) -> list:
    """Reproject one monthly field to the GPW grid; population-weighted
    mean per state."""
    fine = (field.rename({"latitude": "y", "longitude": "x"})
                 .rio.write_crs("EPSG:4326")
                 .rio.write_nodata(np.nan)
                 .rio.reproject_match(pop, resampling=Resampling.bilinear)
                 .values.astype("float64"))
    fine[~np.isfinite(fine)] = np.nan
    valid = np.isfinite(fine)
    w = np.where(valid, pop_np, 0.0)
    num = np.bincount(label.ravel(),
                      weights=(w * np.nan_to_num(fine)).ravel(),
                      minlength=len(state_names) + 1)
    den = np.bincount(label.ravel(), weights=w.ravel(),
                      minlength=len(state_names) + 1)
    return [num[i + 1] / den[i + 1] if den[i + 1] > 0 else np.nan
            for i in range(len(state_names))]


# ----------------------------------------------------------------------
# 4. Cache: which state-months are already done?
# ----------------------------------------------------------------------


clim = None
if CLIM_CACHE.exists() and CACHE_META.exists():
    if json.loads(CACHE_META.read_text()) == CACHE_PARAMS:
        clim = pd.read_csv(CLIM_CACHE)
        for v in CLIM_VARS:
            if v not in clim.columns:
                clim[v] = np.nan
    else:
        print("Base temperatures changed -- discarding stale cache.")
if clim is None:
    clim = pd.DataFrame(columns=["state_name", "year", "month"] + CLIM_VARS)

cached_ym = set(zip(clim["year"].astype(int), clim["month"].astype(int))) \
    if len(clim) else set()
# months whose humid variables are already filled (all states non-null)
humid_ok = set()
# months whose t2m-only variables are already filled (catches caches built
# before t_mean existed: those months are in cached_ym but need recomputing)
t2m_ok = set()
if len(clim):
    g = clim.groupby(["year", "month"])[HUMID_VARS].apply(
        lambda d: d.notna().all().all())
    humid_ok = {(int(y), int(m)) for (y, m), ok in g.items() if ok}
    g2 = clim.groupby(["year", "month"])[T2M_ONLY_VARS].apply(
        lambda d: d.notna().all().all())
    t2m_ok = {(int(y), int(m)) for (y, m), ok in g2.items() if ok}

# ----------------------------------------------------------------------
# 5. Loop over t2m chunks; compute only what's missing
# ----------------------------------------------------------------------
new_records = []
pending_d2m = []                          # panel months awaiting dewpoint

for pt in sorted(ERA5_DIR.glob("t2m_daily_mean_*.nc")):
    m = FNAME_RE.match(pt.name)
    if not m:
        continue
    year, m0, m1 = int(m[1]), int(m[2]), int(m[3])
    file_yms = [(year, mm) for mm in range(m0, m1 + 1)
                if (year, mm) in panel_month_set]
    if not file_yms:
        continue

    pdw = pt.with_name("d2m" + pt.name[3:])
    have_d2m = pdw.exists() and pdw.stat().st_size > 10_000
    if not have_d2m:
        pending_d2m += file_yms

    todo = [ym for ym in file_yms
            if ym not in cached_ym
            or ym not in t2m_ok
            or (have_d2m and ym not in humid_ok)]
    if not todo:
        print(f"skip   {pt.name} (all months cached)")
        continue

    print(f"open   {pt.name}" + ("" if have_d2m else "   [no d2m yet]"))
    fields = monthly_fields(pt, pdw if have_d2m else None)

    for t in fields["cdd"]["time"].values:
        ts = pd.Timestamp(t)
        ym = (ts.year, ts.month)
        if ym not in todo:
            continue
        vals = {v: (weighted_state_means(fields[v].sel(time=t))
                    if v in fields else [np.nan] * len(state_names))
                for v in CLIM_VARS}
        for i, name in enumerate(state_names):
            new_records.append({"state_name": name, "year": ts.year,
                                "month": ts.month,
                                **{v: vals[v][i] for v in CLIM_VARS}})
        print(f"  {ts.year}-{ts.month:02d}: done"
              + ("" if have_d2m else " (t2m-only)"))

# Replace recomputed months in the cache, append new ones
if new_records:
    new_df = pd.DataFrame(new_records)
    redone = set(zip(new_df["year"], new_df["month"]))
    keep = ~clim.apply(lambda r: (int(r["year"]), int(r["month"])) in redone,
                       axis=1) if len(clim) else []
    clim = pd.concat([clim[keep] if len(clim) else clim, new_df],
                     ignore_index=True)
    clim = clim.sort_values(["year", "month", "state_name"])
    clim.to_csv(CLIM_CACHE, index=False)
    CACHE_META.write_text(json.dumps(CACHE_PARAMS))
    print(f"\nCache updated: {CLIM_CACHE} ({len(clim)} state-month rows)")
else:
    print("\nNothing to compute -- cache already up to date.")

# ----------------------------------------------------------------------
# 6. Status report and merge onto the panel
# ----------------------------------------------------------------------
done_ym = set(zip(clim["year"].astype(int), clim["month"].astype(int)))
missing_t2m = sorted(panel_month_set - done_ym)
if missing_t2m:
    print(f"WARNING: no t2m data yet for {len(missing_t2m)} panel months: "
          f"{[f'{y}-{mm:02d}' for y, mm in missing_t2m]}")
if pending_d2m:
    print(f"NOTE: {len(pending_d2m)} panel months have t2m variables only "
          "(d2m not downloaded yet) -- re-run this cell once 01 finishes "
          "and it will fill in cdd_swbgt / cdd_dewpt / rh.")

out = panel.merge(clim, on=["state_name", "year", "month"], how="left")
for v in CLIM_VARS:
    n = out[v].isna().sum()
    if n:
        print(f"  {v}: {n} panel rows missing")

out.to_csv(OUT_CSV, index=False)
print(f"\nSaved: {OUT_CSV.resolve()}  ({len(out)} rows)")

zero_share = (out["cdd_dewpt"] == 0).mean()
print(f"Share of state-months with zero dew-point CDD: {zero_share:.1%} "
      "(if very high, lower DEWPOINT_BASE_C)")

Panel: 33 states, 133 months
State geometries ready: 33
Population grid: (3412, 3628), total = 1.889 bn
Base temperatures changed -- discarding stale cache.
open   t2m_daily_mean_2015_04-12.nc
  2015-04: done
  2015-05: done
  2015-06: done
  2015-07: done
  2015-08: done
  2015-09: done
  2015-10: done
  2015-11: done
  2015-12: done
open   t2m_daily_mean_2016_01-12.nc
  2016-01: done
  2016-02: done
  2016-03: done
  2016-04: done
  2016-05: done
  2016-06: done
  2016-07: done
  2016-08: done
  2016-09: done
  2016-10: done
  2016-11: done
  2016-12: done
open   t2m_daily_mean_2017_01-12.nc
  2017-01: done
  2017-02: done
  2017-03: done
  2017-04: done
  2017-05: done
  2017-06: done
  2017-07: done
  2017-08: done
  2017-09: done
  2017-10: done
  2017-11: done
  2017-12: done
open   t2m_daily_mean_2018_01-12.nc
  2018-01: done
  2018-02: done
  2018-03: done
  2018-04: done
  2018-05: done
  2018-06: done
  2018-07: done
  2018-08: done
  2018-09: done
  2018-10: done
  2018-11: 